## **Structured Output**
---
**Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.**

### **Pydantic**
**Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.**

In [5]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b") 
model

ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019D2A3A0290>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019D2B417350>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [14]:
model.invoke("Who is Cristiano Ronaldo?")

AIMessage(content='<think>\nOkay, so I need to explain who Cristiano Ronaldo is. Let me start by recalling what I know. He\'s a famous footballer, right? Portuguese, I think. Plays for a club in Italy, maybe Juventus? Or has he moved to another team? Wait, I remember he was with Real Madrid for a long time. Real Madrid is a top Spanish club. Then maybe he went to Manchester United after that. Hmm, not sure about the exact timeline.\n\nHe\'s known as one of the best players in the world. I\'ve heard he\'s won multiple Ballon d\'Or awards. The Ballon d\'Or is given to the best male footballer each year. I think he\'s won it a few times. Maybe five? I should check how many. Also, he\'s been in the news for his performances in the UEFA Champions League. He\'s scored a lot of goals there, maybe the all-time top scorer?\n\nI remember he was born in Portugal. His full name is Cristiano Ronaldo dos Santos Aveiro. I think he started his professional career in Portugal with Sporting Lisbon, then

#### **Implenting Strctured Output**

In [11]:
from pydantic import BaseModel, Field

class FootabllPlayer(BaseModel):
    name: str = Field(description="The name of the football player")
    country: str = Field(description="The country the football player is from")
    position: str = Field(description="The position the football player plays")
    goals: int = Field(description="The number of goals the football player has scored in whole career")

In [12]:
model_with_structured_output = model.with_structured_output(FootabllPlayer)
response = model_with_structured_output.invoke("Tell me about Cristiano Ronaldo")

In [13]:
response

FootabllPlayer(name='Cristiano Ronaldo', country='Portugal', position='Forward', goals=851)

#### **Getting Raw AIMessage Along With Parsed Output**

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)  ## include_raw : This option allows you to include the raw response from the model in the output, in addition to the structured data. This can be useful for debugging or for cases where you want to see the original response alongside the structured interpretation.

response = model_with_structured_output.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The tools provided include a Movie function that requires title, year, director, and rating. I need to fill those in for Inception.\n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me double-check that. Yep, IMDb does give it an 8.8. So I should structure the tool call with those parameters. Make sure all required fields are included. No need for any other functions since the user just wants movie details. Alright, that should cover it.\n', 'tool_calls': [{'id': 'tp599ndwq', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 231, 

#### **Nested Structure**

In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response 

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Science Fiction', 'Action'], budget=160.0)

### **Structured Output With TypeDict**

In [17]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [18]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [19]:
## We can also see the model profile
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

#### **Structured Output Using Dataclass**
**A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the `@dataclass` decorator**

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='fb51e18b-f694-4741-8775-7381a67f553b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". Let me break this down.\n\nFirst, I need to identify the different parts. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is "(555) 123-4567". \n\nNow, looking at the provided function, ContactInfo, the required parameters are name, email, and phone. All three are present here. I need to make sure the phone number is in the correct format. The function\'s example includes a phone number with parentheses and hyphens, so the user\'s input matches that format. \n\nI should check if there are any optional parameters, but the required ones are all there. No extr

In [21]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [23]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [24]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')